#### What we will use
1. IAM roles and users
2. S3 buckets
3. Complete Infrastracture of AWS Sagemaker - Training, Endpoints


In [18]:
import os

os.environ["OS_OPT"] = "linux"

import sagemaker
from sagemaker.core.helper.session_helper import Session
from sklearn.model_selection import train_test_split
import boto3
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
role = os.getenv("SAGEMAKER_ROLE_ARN")

sm_boto3 = boto3.client("sagemaker")
sess = Session()
region = sess.boto_session.region_name
bucket = "mobbucketsagemaker123321"
print("Using bucket" + bucket)

Using bucketmobbucketsagemaker123321


In [19]:
print(region)

eu-west-3


In [20]:
df = pd.read_csv("mob_price_classification_train.csv")
df.head()

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1


df.shape

In [21]:
df.isnull().sum()

battery_power    0
blue             0
clock_speed      0
dual_sim         0
fc               0
four_g           0
int_memory       0
m_dep            0
mobile_wt        0
n_cores          0
pc               0
px_height        0
px_width         0
ram              0
sc_h             0
sc_w             0
talk_time        0
three_g          0
touch_screen     0
wifi             0
price_range      0
dtype: int64

In [22]:
df['price_range'].value_counts()

price_range
1    500
2    500
3    500
0    500
Name: count, dtype: int64

In [23]:
df.columns

Index(['battery_power', 'blue', 'clock_speed', 'dual_sim', 'fc', 'four_g',
       'int_memory', 'm_dep', 'mobile_wt', 'n_cores', 'pc', 'px_height',
       'px_width', 'ram', 'sc_h', 'sc_w', 'talk_time', 'three_g',
       'touch_screen', 'wifi', 'price_range'],
      dtype='object')

In [24]:
features = list(df.columns)
features

['battery_power',
 'blue',
 'clock_speed',
 'dual_sim',
 'fc',
 'four_g',
 'int_memory',
 'm_dep',
 'mobile_wt',
 'n_cores',
 'pc',
 'px_height',
 'px_width',
 'ram',
 'sc_h',
 'sc_w',
 'talk_time',
 'three_g',
 'touch_screen',
 'wifi',
 'price_range']

In [25]:
label = features.pop(-1)
label

'price_range'

In [26]:
x=df[features]
y=df[label]

In [27]:
X_train, X_test, y_train, y_test = train_test_split(x,y,test_size=0.15,random_state=0)


In [28]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(1700, 20)
(300, 20)
(1700,)
(300,)


In [29]:
trainX = pd.DataFrame(X_train)
trainX[label] = y_train

testX=pd.DataFrame(X_test)
testX[label] = y_test

In [30]:
trainX

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
1452,1450,0,2.1,0,1,0,31,0.6,114,5,...,1573,1639,794,11,5,9,0,1,1,1
1044,1218,1,2.8,1,3,0,39,0.8,150,7,...,1122,1746,1667,10,0,12,0,0,0,1
1279,1602,0,0.6,0,12,0,58,0.4,170,1,...,1259,1746,3622,17,2,17,0,1,1,3
674,1034,0,2.6,1,2,1,45,0.3,190,3,...,182,1293,969,15,1,7,1,0,0,0
1200,530,0,2.4,0,1,0,32,0.3,88,6,...,48,1012,959,17,7,6,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
835,1224,1,1.6,0,9,0,33,1.0,157,1,...,522,563,3796,10,5,13,1,1,0,3
1216,1158,0,0.7,1,1,1,29,0.7,123,2,...,311,1796,1542,17,9,15,1,0,1,1
1653,1190,0,2.0,1,0,0,40,0.2,93,5,...,1399,1646,3610,13,7,9,0,0,1,3
559,1191,0,2.4,1,2,0,13,0.9,169,1,...,179,1813,1028,14,6,8,1,1,1,0


In [31]:
trainX.to_csv("train-V-1.csv", index=False)
testX.to_csv("test-V-1.csv", index=False)

In [32]:
bucket

'mobbucketsagemaker123321'

In [33]:
## send data to S3. Sagemkaer will take the data for training from s3
sk_prefix="sagemaker/mobile_price_classification/sklearncontainer"
trainpath=sess.upload_data(path='train-V-1.csv', bucket=bucket, key_prefix = sk_prefix)
testpath=sess.upload_data(path='test-V-1.csv', bucket=bucket, key_prefix = sk_prefix)
print(trainpath)
print(testpath)

s3://mobbucketsagemaker123321/sagemaker/mobile_price_classification/sklearncontainer/train-V-1.csv
s3://mobbucketsagemaker123321/sagemaker/mobile_price_classification/sklearncontainer/test-V-1.csv


## Script used by AWS Sagemaker to train models

In [34]:
%%writefile src/script.py

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score
import sklearn
import joblib
import boto3
import pathlib
from io import StringIO
import argparse
import os
import numpy as np
import pandas as pd

def model_fn(model_dir):
    clf = joblib.load(os.path.join(model_dir, "model.joblib"))
    return clf


if __name__=="__main__":
    print("[Info] Extracting arguments")
    parser = argparse.ArgumentParser()

    # Hyperparameter
    parser.add_argument("--n-estimators", type=int, default=100)
    parser.add_argument("--random-state", type=int, default=0)

    ### Data, model and output directories
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--test", type=str, default=os.environ.get("SM_CHANNEL_TEST"))
    parser.add_argument("--train-file", type=str, default="train-V-1.csv")
    parser.add_argument("--test-file", type=str, default="test-V-1.csv")

    args, _ = parser.parse_known_args()

    print("SKLearn Version: ", sklearn.__version__)
    print("Jobliv Version: ", joblib.__version__)

    print("[INFO] Reading data")
    print()
    train_df = pd.read_csv(os.path.join(args.train, args.train_file))
    test_df = pd.read_csv(os.path.join(args.test, args.test_file))

    features = list(train_df.columns)
    label = features.pop(-1)

    print("Building training and testing datasets")
    print()
    X_train = train_df[features]
    X_test = test_df[features]
    y_train = train_df[label]
    y_test = test_df[label]

    print('Column order: ')
    print(features)
    print()

    print("Label column is: ", label)
    print()

    print("Data shape: ")
    print()
    print("Shape of training data (85%):")
    print(X_train.shape)
    print(y_train.shape)
    print()
    print("Shape of testing data (15%):")
    print(X_test.shape)
    print(y_test.shape)
    print()

    print("Training RandomForest Model...")
    print()
    model=RandomForestClassifier(n_estimators=args.n_estimators, random_state=args.random_state,\
                                 verbose=2, n_jobs=1)

    model.fit(X_train,y_train)

    print()

    model_path = os.path.join(args.model_dir, "model.joblib")
    joblib.dump(model, model_path)

    print("Model saved at " + model_path)

    y_pred_test = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred_test)
    test_rep = classification_report(y_test, y_pred_test)

    print()
    print("Metrics on test")
    print("Total rows are: ", X_test.shape[0])
    print('[TESTING] Model accuracy is: ', test_acc)
    print('[TESTING] Testing report: ')
    print(test_rep)
    


    



Overwriting src/script.py


### AWS Sagemaker Entry point to Execute the Training script

In [36]:
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import (
    Compute,
    SourceCode,
    StoppingCondition,
)



# 1. Retrieve the Scikit-learn ECR image URI using sess
sess = Session()
image_uri = image_uris.retrieve(
    framework="sklearn",
    region=sess.boto_region_name,
    version="1.2-1",
    py_version="py3",
    instance_type="ml.m5.large"
)

# 2. Configure Compute hardware & enable spot training
compute_config = Compute(
    instance_type="ml.m5.large",
    instance_count=1,
    enable_managed_spot_training=True,
)

# 3. Configure StoppingCondition timeouts
stopping_config = StoppingCondition(
    max_runtime_in_seconds=3600,     # Max training time: 1 hour
    max_wait_time_in_seconds=7200,    # Max total spot wait time: 2 hours
)


# 4. Configure Source Code (source_dir is required when entry_script is provided)
source_config = SourceCode(
    source_dir="src",                   # Current directory where script.py lives
    entry_script="script.py"
)

# 5. Instantiate ModelTrainer
sklearn_trainer = ModelTrainer(
    training_image=image_uri,
    role=role,
    source_code=source_config,
    compute=compute_config,
    stopping_condition=stopping_config,
    base_job_name="RF-custom-sklearn",
    hyperparameters={
        "n_estimators": "100",
        "random_state": "0",
    },
)

# To launch training in v3:
# from sagemaker.train.configs import InputData
# train_data = InputData(channel_name="train", data_source="s3://your-bucket/train.csv")
# sklearn_trainer.train(input_data_config=[train_data])

[07/31/26 16:44:57] INFO     SageMaker session not provided. Using default Session.                  ]8;id=12399300;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=12399301;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/train/defaults.py#65\65]8;;\

[07/31/26 16:44:58] INFO     Role 'arn:aws:iam::414069442977:role/sagemakeraccess'         ]8;id=12399308;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=12399309;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/helper/iam_role_resolver.py#598\598]8;;\
                             validated for training. Using it.                                                     

[07/31/26 16:45:00] INFO     OutputDataConfig not provided. Using default:                          ]8;id=12399315;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=12399316;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/train/defaults.py#192\192]8;;\
                             s3_output_path='s3://sagemaker-eu-west-3-414069442977/RF-custom-sklear                
                             n' kms_key_id=None compression_type='GZIP'                                            

                    INFO     Training image URI:                                               ]8;id=12399323;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=12399324;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             659782779980.dkr.ecr.eu-west-3.amazonaws.com/sagemaker-scikit-lea                     
                             rn:1.2-1-cpu-py3                                                                      

In [37]:
from sagemaker.train.configs import Compute, StoppingCondition

print("Compute fields:", list(Compute.model_fields.keys()))
print("StoppingCondition fields:", list(StoppingCondition.model_fields.keys()))

Compute fields: ['instance_type', 'instance_count', 'volume_size_in_gb', 'volume_kms_key_id', 'keep_alive_period_in_seconds', 'instance_groups', 'training_plan_arn', 'instance_placement_config', 'enable_managed_spot_training']
StoppingCondition fields: ['max_runtime_in_seconds', 'max_wait_time_in_seconds', 'max_pending_time_in_seconds']


In [39]:
# launch training
from sagemaker.train.configs import InputData
inputs = [
    InputData(channel_name="train", data_source=trainpath),
    InputData(channel_name="test", data_source=testpath),
]
sklearn_trainer.train(input_data_config=inputs)

[07/31/26 16:45:14] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=12399331;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=12399332;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/telemetry/telemetry_logging.py#304\304]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/tiziana/.config/sagemaker/config.yaml


[07/31/26 16:45:19] INFO     Creating training_job resource.                                     ]8;id=12399339;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12399340;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/resources.py#31239\31239]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=12399347;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=12399348;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/utils/utils.py#361\361]8;;\

[07/31/26 16:45:20] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=12399353;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=12399354;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/botocore/credentials.py#1392\1392]8;;\

...

                    INFO     RF-custom-sklearn-20260731164514/algo-1-1785509098:                 ]8;id=12400668;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12400669;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/resources.py#31585\31585]8;;\
                             macro avg       0.88      0.88      0.88       300                                    

                    INFO     RF-custom-sklearn-20260731164514/algo-1-1785509098:                 ]8;id=12400674;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12400675;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/resources.py#31585\31585]8;;\
                             weighted avg       0.88      0.88      0.88       300                                 

                    INFO     RF-custom-sklearn-20260731164514/algo-1-1785509098:                 ]8;id=12400680;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12400681;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/resources.py#31585\31585]8;;\
                             ++ echo 'Training Container Execution Completed'                                      

                    INFO     RF-custom-sklearn-20260731164514/algo-1-1785509098:                 ]8;id=12400686;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12400687;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/resources.py#31585\31585]8;;\
                             Training Container Execution Completed                                                

[07/31/26 16:47:58] INFO     Final Resource Status: Completed                                    ]8;id=12400693;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12400694;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/resources.py#31591\31591]8;;\

In [40]:
import boto3

# Get the exact training job name from SageMaker v3 TrainingJob object
job_name = sklearn_trainer._latest_training_job.training_job_name
print(f"Fetching CloudWatch logs for Training Job: {job_name}\n")

# Initialize CloudWatch Logs client
logs_client = boto3.client("logs")
log_group = "/aws/sagemaker/TrainingJobs"

try:
    # Fetch log streams for this job
    streams_response = logs_client.describe_log_streams(
        logGroupName=log_group, logStreamNamePrefix=job_name
    )
    streams = streams_response.get("logStreams", [])

    if not streams:
        print("No log streams found for this job yet.")
    else:
        # Loop through streams and output log messages
        for stream in streams:
            events_response = logs_client.get_log_events(
                logGroupName=log_group,
                logStreamName=stream["logStreamName"],
                limit=100,
            )
            for event in events_response.get("events", []):
                print(event["message"].strip())

except Exception as e:
    print(f"Error fetching CloudWatch logs: {e}")

Fetching CloudWatch logs for Training Job: RF-custom-sklearn-20260731164514

building tree 20 of 100
building tree 21 of 100
building tree 22 of 100
building tree 23 of 100
building tree 24 of 100
building tree 25 of 100
building tree 26 of 100
building tree 27 of 100
building tree 28 of 100
building tree 29 of 100
building tree 30 of 100
building tree 31 of 100
building tree 32 of 100
building tree 33 of 100
building tree 34 of 100
building tree 35 of 100
building tree 36 of 100
building tree 37 of 100
building tree 38 of 100
building tree 39 of 100
building tree 40 of 100
[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:    0.1s
building tree 41 of 100
building tree 42 of 100
building tree 43 of 100
building tree 44 of 100
building tree 45 of 100
building tree 46 of 100
building tree 47 of 100
building tree 48 of 100
building tree 49 of 100
building tree 50 of 100
building tree 51 of 100
building tree 52 of 100
building tree 53 of 100
building tree 54 of 100
building tree 55 of 10

##  To get the model from S3

In [ ]:
sklearn_trainer._latest_training_job.wait(logs=False)

artifact = sm_boto3.describe_training_job(
    TrainingJobName=sklearn_trainer._latest_training_job.training_job_name
)["ModelArtifacts"]["S3ModelArtifacts"]

[07/31/26 16:52:00] INFO     Final Resource Status: Completed                                    ]8;id=12400699;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12400700;file:///home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/sagemaker/core/resources.py#31591\31591]8;;\

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:4                                                                                    │
│                                                                                                  │
│   1 sklearn_trainer._latest_training_job.wait(logs=False)                                        │
│   2                                                                                              │
│   3 artifact = sm_boto3.describe_training_job(                                                   │
│ ❱ 4 │   TrainingJobName=sklearn_trainer._latest_training_job.job_name                            │
│   5 )["ModelArtifacts"]["S3ModelArtifacts"]                                                      │
│   6                                                                                              │
│                                                                                                  │
│ /home/tiziana/miniconda3/envs/awssagemaker/lib/python3.10/site-packages/pydantic/main.py:1042 in │
│ __getattr__                                                                                      │
│                                                                                                  │
│   1039 │   │   │   │   │   │   return super().__getattribute__(item)  # Raises AttributeError i  │
│   1040 │   │   │   │   │   else:                                                                 │
│   1041 │   │   │   │   │   │   # this is the current error                                       │
│ ❱ 1042 │   │   │   │   │   │   raise AttributeError(f'{type(self).__name__!r} object has no att  │
│   1043 │   │                                                                                     │
│   1044 │   │   def __setattr__(self, name: str, value: Any) -> None:                             │
│   1045 │   │   │   if (setattr_handler := self.__pydantic_setattr_handlers__.get(name)) is not   │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
AttributeError: 'TrainingJob' object has no attribute 'job_name'

In [ ]:
artifact